[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_1_2/05_tag_1_2_training_loss_regularisierung.ipynb)

# Tag 1-2 - 05 Training, Loss, Bias-Variance und Regularisierung

Regularisierung erkennt Ausreißer nicht magisch. Sie bestraft große Gewichte. Bei flexiblen Modellen mit Ausreißern kann das verhindern, dass das Modell den Ausreißern hinterherläuft.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
plt.rcParams['figure.figsize'] = (7, 4)
plt.rcParams['axes.grid'] = True
print('Setup abgeschlossen.')


## Loss-Funktionen und manuelles Parameter-Testen


In [ ]:
def loss_values(y_true, y_pred, loss='MSE'):
    errors = y_true - y_pred
    if loss == 'MAE / L1':
        return np.mean(np.abs(errors))
    return np.mean(errors**2)

def fit_line_closed_form(x_values, y_values):
    X_design = np.c_[np.ones(len(x_values)), x_values]
    b_learned, m_learned = np.linalg.lstsq(X_design, y_values, rcond=None)[0]
    return m_learned, b_learned

def model_loss_demo(samples=80, rauschen=0.7, ausreisser=4, ausreisser_staerke=8.0, m=2.0, b=0.0, loss='MSE'):
    local_rng = np.random.default_rng(RANDOM_STATE)
    x_values = np.linspace(-3, 3, samples)
    y_clean = 1.8 * x_values - 0.5
    y = y_clean + local_rng.normal(0, rauschen, samples)
    outlier_idx = np.array([], dtype=int)
    if ausreisser > 0:
        outlier_idx = local_rng.choice(samples, size=min(ausreisser, samples), replace=False)
        y[outlier_idx] += ausreisser_staerke

    manual_pred = m * x_values + b
    learned_m, learned_b = fit_line_closed_form(x_values, y)
    learned_pred = learned_m * x_values + learned_b
    manual_loss = loss_values(y, manual_pred, loss)
    learned_loss = loss_values(y, learned_pred, loss)

    m_grid = np.linspace(-2.5, 5.5, 180)
    loss_curve = np.array([loss_values(y, m_candidate * x_values + b, loss) for m_candidate in m_grid])

    fig, axes = plt.subplots(1, 3, figsize=(17, 4))
    axes[0].scatter(x_values, y, alpha=0.75, label='Daten')
    if len(outlier_idx):
        axes[0].scatter(x_values[outlier_idx], y[outlier_idx], color='red', s=80, label='Ausreißer')
    axes[0].plot(x_values, y_clean, 'g--', label='wahre Funktion')
    axes[0].plot(x_values, manual_pred, color='black', label=f'manuell: y={m:.1f}x+{b:.1f}')
    axes[0].plot(x_values, learned_pred, color='orange', linewidth=2, label=f'gelernt: y={learned_m:.2f}x+{learned_b:.2f}')
    axes[0].set_title('Manuelle und gelernte Funktion')
    axes[0].legend()

    axes[1].plot(m_grid, loss_curve)
    axes[1].scatter([m], [manual_loss], color='black', s=80, label='deine Einstellung')
    learned_loss_at_fixed_b = loss_values(y, learned_m * x_values + b, loss)
    axes[1].scatter([learned_m], [learned_loss_at_fixed_b], color='orange', s=80, label='gelernte Steigung bei deinem b')
    axes[1].set_xlabel('Steigung m bei festem b')
    axes[1].set_ylabel(loss)
    axes[1].set_title('Loss-Funktion entlang m')
    axes[1].legend()

    max_loss = max(manual_loss, learned_loss, np.percentile(loss_curve, 95))
    axes[2].bar(['deine Funktion', 'gelerntes Minimum'], [manual_loss, learned_loss], color=['black', 'orange'])
    axes[2].set_ylim(0, max_loss * 1.15)
    axes[2].set_ylabel(loss)
    axes[2].set_title('Loss-Vergleich')
    plt.tight_layout()
    plt.show()

widgets.interact(
    model_loss_demo,
    samples=widgets.IntSlider(value=80, min=30, max=200, step=10),
    rauschen=widgets.FloatSlider(value=0.7, min=0.0, max=3.0, step=0.1),
    ausreisser=widgets.IntSlider(value=4, min=0, max=20),
    ausreisser_staerke=widgets.FloatSlider(value=8.0, min=-15.0, max=20.0, step=0.5),
    m=widgets.FloatSlider(value=2.0, min=-2.0, max=5.0, step=0.1),
    b=widgets.FloatSlider(value=0.0, min=-5.0, max=5.0, step=0.1),
    loss=widgets.Dropdown(options=['MSE', 'MAE / L1'], value='MSE'),
);


## Regularisierung mit Ausreißern: flexibles Modell stabilisieren

Die hier verwendete Zielfunktion ist:

`J(w) = MSE(y, y_hat) + alpha * Summe(w_i^2)`

Der Achsenabschnitt wird nicht bestraft. Regularisierung erkennt Ausreißer nicht direkt, sondern macht große Gewichte teuer. Bei einem flexiblen Polynommodell verhindert das, dass die Kurve Ausreißern extrem hinterherläuft.


In [ ]:
def true_function(x_values):
    return 0.5 + 1.2 * x_values - 0.7 * x_values**2 + 0.35 * x_values**3

def design(x_values, degree):
    x_values = np.asarray(x_values)
    return np.vstack([x_values ** d for d in range(degree + 1)]).T

def fit_ridge(x_values, y_values, degree, alpha):
    X_design = design(x_values, degree)
    penalty = np.eye(X_design.shape[1])
    penalty[0, 0] = 0
    return np.linalg.solve(X_design.T @ X_design + alpha * penalty, X_design.T @ y_values)

def pred_ridge(x_values, weights):
    return design(x_values, len(weights) - 1) @ weights

def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2)))

def regularization_demo(alpha=1.0, polynomgrad=14, ausreisser=5, ausreisser_staerke=7.0):
    local_rng = np.random.default_rng(RANDOM_STATE)
    x_train = np.sort(local_rng.uniform(-2.4, 2.4, 45))
    y_train = true_function(x_train) + local_rng.normal(0, 0.45, len(x_train))
    out_idx = np.array([], dtype=int)
    if ausreisser > 0:
        out_idx = local_rng.choice(len(x_train), size=min(ausreisser, len(x_train)), replace=False)
        y_train[out_idx] += ausreisser_staerke * local_rng.choice([-1, 1], size=len(out_idx))
    x_dense = np.linspace(-2.6, 2.6, 350)
    y_true_dense = true_function(x_dense)

    w_no_reg = fit_ridge(x_train, y_train, polynomgrad, alpha=0.0)
    w_reg = fit_ridge(x_train, y_train, polynomgrad, alpha=alpha)
    pred_no_reg = pred_ridge(x_dense, w_no_reg)
    pred_reg = pred_ridge(x_dense, w_reg)

    rows = pd.DataFrame({
        'Modell': ['ohne Regularisierung', f'L2 alpha={alpha}'],
        'Train RMSE': [rmse(y_train, pred_ridge(x_train, w_no_reg)), rmse(y_train, pred_ridge(x_train, w_reg))],
        'RMSE gegen wahre Funktion': [rmse(y_true_dense, pred_no_reg), rmse(y_true_dense, pred_reg)],
        'Gewichtsnorm': [np.linalg.norm(w_no_reg[1:]), np.linalg.norm(w_reg[1:])],
    })
    display(rows.round(3))

    plt.scatter(x_train, y_train, alpha=0.75, label='Training')
    if len(out_idx):
        plt.scatter(x_train[out_idx], y_train[out_idx], color='red', s=80, label='Ausreißer')
    plt.plot(x_dense, y_true_dense, 'g--', linewidth=2, label='wahre Funktion')
    plt.plot(x_dense, pred_no_reg, color='#e45756', linewidth=2, label='ohne Regularisierung')
    plt.plot(x_dense, pred_reg, color='black', linewidth=2, label=f'L2 alpha={alpha}')
    plt.ylim(-9, 9)
    plt.xlabel('x')
    plt.ylabel('y')
    plt.title('Regularisierung verhindert extremes Hinterherlaufen')
    plt.legend()
    plt.show()

widgets.interact(
    regularization_demo,
    alpha=widgets.FloatLogSlider(value=1.0, base=10, min=-3, max=3, step=0.25),
    polynomgrad=widgets.IntSlider(value=14, min=3, max=20),
    ausreisser=widgets.IntSlider(value=5, min=0, max=15),
    ausreisser_staerke=widgets.FloatSlider(value=7.0, min=0.0, max=12.0, step=0.5),
);
